# Fold illusions with factorized blending

The same illusion as [`illusions.ipynb`](illusions.ipynb), with the same prompts: a city street at night when the sheet lies flat, and a fox and an owl on the folded crane's two wings. Only the flat sheet and the finished crane are used, with no intermediate stages.

**The one change is how visible pixels are shared.** In the original, a pixel that shows on the folded crane follows two prompts at once, and the sampler averages their predictions. Following [Factorized Diffusion](https://dangeng.github.io/factorized_diffusion/) (Geng, Park, Owens, ECCV 2024), this notebook splits the shared pixels by frequency instead:

- **The flat prompt** controls the sheet's low frequencies: its broad shapes and colors.
- **The fox and owl prompts** control the high frequencies, wherever their paper shows.

A wing shows only a small patch of the sheet, so its whole picture is fine detail at the sheet's scale. That's why this split might suit folding. The original notebook and its code are untouched: `factorized.py` runs `illusion.generate` unchanged and swaps only its blending step.

**Status:** untested on a GPU. The blend's math, and a full run of the sampler with a stand-in model, are checked by `tests/test_factorized.py`.

## Setup

In [ ]:
import importlib, os, subprocess, sys

FOLDER = "fold-map"
if not os.path.exists("factorized.py"):
    if not os.path.exists("folded-illusions"):
        # The repo is private. On Colab, add a GitHub token as a secret named GITHUB_TOKEN (key icon, left sidebar).
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = f"https://{token}@github.com/rath-hy/folded-illusions.git"
        subprocess.run(["git", "clone", "-q", "--depth", "1", url], check=True)
    os.chdir(os.path.join("folded-illusions", FOLDER))
if "google.colab" in sys.modules:
    # On Colab, rerunning this cell gets the latest code and reloads anything already imported.
    subprocess.run(["git", "pull", "-q"], check=True)
    for name in ["fold_map", "decal", "illusion", "factorized"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from fold_map import FoldedModel
from decal import test_picture
from illusion import FoldViews, canvas_picture, plot_result, printable, square_picture
import factorized

## The crane and its views

As in the original: the flat sheet, plus the folded crane seen from each side. The code calls those views front and back, and each shows one wing. Stage 1 works at 64 × 64, stage 2 at 256 × 256.

In [ ]:
model = FoldedModel("004_traditional_Crane_cp.fold", "004_traditional_Crane_state.fold")
views64, views256 = FoldViews(model, 64), FoldViews(model, 256)
for name, v in (("64", views64), ("256", views256)):
    print(name, v.stats())

## What the split does, without AI

Before spending GPU time, here's the same split applied to finished pictures. The letters are stand-ins: A for the flat picture, and F and B for the two wings.

`SIGMA` is the blur's width as a fraction of the sheet's side, and it's where the split happens. A small blur gives the flat picture almost everything, leaving the folded pictures only their sharpest edges. A larger one leaves the flat picture only the broad shapes, and gives the folded pictures more of their own color and shading. For scale, the crane's whole folded picture spans about 16 × 22 pixels at stage 1, about a quarter of the sheet's side.

Read each row left to right: the flat sheet, then each folded view.

In [ ]:
flat_pic = square_picture(test_picture("A", (60, 140, 90), (1000, 1000)), 256)
front_pic = canvas_picture(views256, test_picture("F", (196, 120, 60), (1300, 1000)), "front")
back_pic = canvas_picture(views256, test_picture("B", (52, 98, 168), (1300, 1000)), "back")

from illusion import composite
previews = {"composite (no blending)": composite(views256, flat_pic, front_pic, back_pic)}
for sigma in (1 / 64, 1 / 32, 1 / 16):
    previews[f"split at SIGMA = 1/{round(1 / sigma)}"] = factorized.preview(views256, flat_pic, front_pic, back_pic, sigma)
factorized.plot_comparison(views256, previews)
plt.show()

In the composite, the flat sheet is covered in sharp shards of the folded pictures. With the split, the shards keep their edges but take the flat picture's colors and shading, so they're harder to spot on the flat sheet. The folded views pay for that: their broad colors come from the flat picture too.

That's the trade we're testing. The generator can do better than this preview, because it draws all three pictures together and can choose content where the trade doesn't show.

## Generate (needs a GPU)

The cells from here to the prompts are the same as in `illusions.ipynb`.

1. Use a **GPU runtime** (Runtime → Change runtime type; a T4 works).
2. On Hugging Face, accept the licenses for [DeepFloyd/IF-I-M-v1.0](https://huggingface.co/DeepFloyd/IF-I-M-v1.0) and [DeepFloyd/IF-II-M-v1.0](https://huggingface.co/DeepFloyd/IF-II-M-v1.0), and have an [access token](https://huggingface.co/settings/tokens) ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece
from huggingface_hub import notebook_login
notebook_login()

### Prompts

The same prompts as `illusions.ipynb`, unchanged, so any difference comes from the blend.

In [ ]:
STYLE = "an oil painting of "
FLAT_PROMPT = STYLE + "a crowded city street at night"
FRONT_PROMPT = STYLE + "a red fox, on a plain light background"
BACK_PROMPT = STYLE + "a white owl, on a plain light background"

### Encode the prompts

Rerun this cell whenever you change the prompts.

In [ ]:
import gc, torch
from transformers import T5EncoderModel
from diffusers import IFPipeline, IFSuperResolutionPipeline

# The safety checker and watermarker aren't used by our sampler, and the safety checker
# doesn't load with current versions of transformers, so skip them.
skip = dict(safety_checker=None, feature_extractor=None, watermarker=None)

text_encoder = T5EncoderModel.from_pretrained("DeepFloyd/IF-I-M-v1.0", subfolder="text_encoder", variant="fp16",
                                              torch_dtype=torch.float16, device_map="cuda")
encoder = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=text_encoder, unet=None, **skip)
prompt_embeds, negative_embeds = encoder.encode_prompt([FLAT_PROMPT, FRONT_PROMPT, BACK_PROMPT])

del encoder, text_encoder
gc.collect(); torch.cuda.empty_cache()
print("embeddings:", tuple(prompt_embeds.shape))

### Load the image models

This only needs to run once per session.

In [ ]:
if "stage_1" not in globals():
    stage_1 = IFPipeline.from_pretrained("DeepFloyd/IF-I-M-v1.0", text_encoder=None, variant="fp16",
                                         torch_dtype=torch.float16, **skip).to("cuda")
    stage_2 = IFSuperResolutionPipeline.from_pretrained("DeepFloyd/IF-II-M-v1.0", text_encoder=None, variant="fp16",
                                                        torch_dtype=torch.float16, **skip).to("cuda")

### Compare the blends

One run with the original averaging, then one per `SIGMA` with the factorized blend, all with the same seed. Everything else is identical, so differences come from the blend alone. Each run takes a minute or two on a T4.

- `SIGMAS`: where the split happens, as a fraction of the sheet's side.
- `LOW`: which prompt gets the low frequencies. `"flat"` is the idea above, and `"folded"` swaps the roles, as a control.
- `FOLD_WEIGHT`: used by the original average, and by both for the variance channels DeepFloyd also predicts.

Stage 1 decides each picture's layout, so its results are shown too.

In [ ]:
from illusion import generate as generate_averaged

SIGMAS = (1 / 64, 1 / 32, 1 / 16)
LOW = "flat"
FOLD_WEIGHT = 0.5
GUIDANCE = (7.0, 4.0)
STEPS = (50, 40)
SEED = 0

embeds = (negative_embeds, prompt_embeds)
settings = dict(guidance=GUIDANCE, steps=STEPS, fold_weight=FOLD_WEIGHT, seed=SEED)
results = {"averaged (original)": generate_averaged(stage_1, stage_2, views64, views256, embeds, **settings)}
for sigma in SIGMAS:
    results[f"factorized, SIGMA = 1/{round(1 / sigma)}"] = factorized.generate(
        stage_1, stage_2, views64, views256, embeds, sigma_frac=sigma, low=LOW, **settings)

factorized.plot_comparison(views64, {k: s1 for k, (s1, _) in results.items()}); plt.suptitle("stage 1 (64 × 64)", y=1.01)
factorized.plot_comparison(views256, {k: s2 for k, (_, s2) in results.items()}); plt.suptitle("stage 2 (256 × 256)", y=1.01)
plt.show()

**What to look for:**

- **Flat:** does the city stay a city, with fewer crane-shaped shards breaking it up?
- **Folded:** do the fox and owl stay readable, especially the owl, which barely showed in the first print?
- **Seams:** creases cut the folded views, and those edges count as high frequency.

Seeds matter as much as settings in these illusions, so compare a few before concluding anything.

### Try more seeds

The original blend and one factorized setting, side by side, for several seeds.

In [ ]:
SEEDS = (1, 2, 3)
SIGMA = 1 / 32

for seed in SEEDS:
    s = dict(settings, seed=seed)
    pair = {f"seed {seed}, averaged": generate_averaged(stage_1, stage_2, views64, views256, embeds, **s)[1],
            f"seed {seed}, factorized 1/{round(1 / SIGMA)}": factorized.generate(
                stage_1, stage_2, views64, views256, embeds, sigma_frac=SIGMA, low=LOW, **s)[1]}
    factorized.plot_comparison(views256, pair)
    plt.show()

### Fold the best one

Pick a result by its label from above. This folds it virtually with the decal simulator, which works independently of the views used to make it, then saves a print file. Print it at **actual size**, and fold with the **printed side out**, as in the original: corner A becomes the tail, C the head, and B and D the wing tips.

In [ ]:
BEST = "factorized, SIGMA = 1/32"     # a key from results, e.g. "averaged (original)"

sheet = results[BEST][1]
plot_result(views256, sheet, BEST)
plt.show()

name = "factorized_" + BEST.split(",")[-1].strip().replace(" ", "").replace("/", "-").replace("=", "") + f"_seed{SEED}.png"
printable(sheet).save(name, dpi=(300, 300))
print("saved", name)
try:
    from google.colab import files
    files.download(name)
except ImportError:
    pass

## Notes

- **The split is linear.** Like Factorized Diffusion, it changes the model's predictions, not the noisy image, so every view still sees properly noisy input.
- **Both stages use the same physical split.** `SIGMA` is a fraction of the sheet, so it's 2 pixels at stage 1 and 8 at stage 2 for 1/32. `factorized.generate(..., factorize=(True, False))` limits it to stage 1.
- **Hidden paper.** Far from any visible paper, the result is the flat prediction alone, as in the original. Within about a blur width of the visible regions, a little of the folded views' detail spreads into hidden paper.
- **Nothing stops leaking.** As Factorized Diffusion notes, one prompt's content can still show up in another's component.

## References

- D. Geng, I. Park, A. Owens. [Factorized Diffusion: Perceptual Illusions by Noise Decomposition](https://dangeng.github.io/factorized_diffusion/). *ECCV*, 2024.
- D. Geng, I. Park, A. Owens. [Visual Anagrams: Generating Multi-View Optical Illusions with Diffusion Models](https://dangeng.github.io/visual_anagrams/). *CVPR*, 2024.
- DeepFloyd. [IF](https://github.com/deep-floyd/IF): a pixel-space cascaded text-to-image diffusion model, 2023.